# Moving Horizon Estimation (MHE): от задачи оптимизации к байесовскому выводу

## 1. Постановка задачи как оптимизационной проблемы

Рассмотрим нелинейную динамическую систему с дискретным временем:

$$
\begin{aligned}
x_{k+1} &= f(x_k, u_k, \theta) + w_k, \quad w_k \sim \mathcal{N}(0, Q), \\
y_k &= h(x_k, \theta) + v_k, \quad v_k \sim \mathcal{N}(0, R),
\end{aligned}
$$

где:
- $x_k \in \mathbb{R}^n$ — состояние,
- $u_k$ — известное управление (может отсутствовать),
- $\theta \in \mathbb{R}^p$ — неизвестные параметры (постоянные или медленно меняющиеся),
- $w_k$, $v_k$ — независимые гауссовские шумы процесса и измерений.

**Задача оценивания:** по конечному набору измерений $y_{k-M+1}, \dots, y_k$ найти наилучшие в смысле максимума правдоподобия (или апостериорной вероятности) оценки состояния $x$ и параметров $\theta$.

При использовании скользящего окна длины $M$ мы приходим к формулировке MHE.

OPAAAAAA

## 3. Учёт априорной информации о параметрах

Если у нас есть априорное знание о параметрах $\theta \sim \mathcal{N}(\bar\theta, P_\theta)$, то к целевой функции добавляется слагаемое:

$$
\frac12 \|\theta - \bar\theta\|_{P_\theta^{-1}}^2.
$$

Это естественно регуляризует задачу и предотвращает переобучение при малом объёме данных.

## 4. Рекурсивное обновление априорной точности (Arrival Cost для параметров)

После обработки очередного окна мы хотим передать информацию о параметрах в следующее окно. Это делается путём обновления обратной ковариационной матрицы (информационной матрицы Фишера) с возможным экспоненциальным забыванием для отслеживания дрейфа:

$$
\Lambda_{k+1} = \lambda \, \Lambda_k + \mathcal{I}_{\text{window}},
$$

где $\Lambda_k = C_{\theta,k}^{-1}$ — точность априорного распределения параметров перед окном $k$, $\lambda \in (0,1]$ — фактор забывания (обычно $0.95\ldots0.995$), $\mathcal{I}_{\text{window}}$ — информация, полученная в текущем окне (часто аппроксимируется матрицей Фишера).

При $\lambda=1$ накопление соответствует байесовскому выводу для постоянных параметров; $\lambda<1$ позволяет адаптироваться к медленным изменениям.

## 5. Матрица Фишера (FIM) и её роль

Информационная матрица Фишера для параметров $\theta$ (при фиксированных начальных условиях и траектории управления) задаётся как:

$$
\mathcal{I}(\theta) = \mathbb{E}_{y}\left[ \left( \frac{\partial \log p(y_{1:N}\mid\theta)}{\partial \theta} \right)^T \left( \frac{\partial \log p(y_{1:N}\mid\theta)}{\partial \theta} \right) \right] 
= \underbrace{\sum_{i=1}^{N} H_i^T R^{-1} H_i}_{\text{от измерений}} +
\underbrace{\sum_{i=0}^{N-1} F_i^T Q^{-1} F_i}_{\text{от динамики}} + P_\theta^{-1},
$$

где $H_i = \frac{\partial h}{\partial \theta}\big|_{x_i,\theta}$, $F_i = \frac{\partial f}{\partial \theta}\big|_{x_i,\theta}$.

На практике FIM можно вычислять:
- **Полную** (с учётом $Q$) – теоретически более точна, но требует якобианов динамики.
- **Наблюдаемую** (только измерения) – проще, часто используется, если шум процесса мал или динамические якобианы нестабильны.

В коде (функции `compute_fim` и `compute_observed_fim`) реализованы оба варианта.

In [ ]:
# Пример вычисления наблюдаемой FIM для простой линейной модели
import numpy as np
import scipy.linalg as la

def compute_observed_fim_example(theta, R_inv, measurements, observation_jacobian):
    """
    Вычисление наблюдаемой FIM по измерениям.
    theta: текущая оценка параметров
    R_inv: обратная ковариационная матрица шума измерений
    measurements: массив измерений (N, ny)
    observation_jacobian: функция, возвращающая список якобианов H_i (size p x ny) для каждого измерения
    """
    F = np.zeros((len(theta), len(theta)))
    for i, y in enumerate(measurements):
        H = observation_jacobian(i, theta)  # p x ny
        F += H @ R_inv @ H.T
    return F

# Демонстрационный запуск (фиктивные данные не выполняем)

### 5.1 Регуляризация FIM

При ограниченном числе данных или плохой обусловленности накопленная информационная матрица может стать вырожденной. Для обеспечения устойчивости применяется регуляризация:

1. **Спектральная:** собственные числа $\lambda_i$ матрицы поднимаются до минимального порога $\tau = \max(\tau_{\text{ratio}} \lambda_{\max}, \tau_{\min})$:
   $$
   \tilde{\Lambda} = V \, \text{diag}(\max(\lambda_i, \tau)) \, V^T.
   $$
2. **Ridge:** добавляется $\beta I$ для улучшения обусловленности.

В коде (функция `regularize_fim`) вы можете комбинировать оба метода.

In [ ]:
def regularize_fim_example(F, tau_ratio=1e-3, min_tau=1e-2, ridge=1.0):
    """
    Регуляризация информационной матрицы.
    Возвращает регуляризованную матрицу и собственные числа.
    """
    F = (F + F.T) / 2.0
    eigvals, eigvecs = la.eigh(F)
    eigvals = eigvals[::-1]
    eigvecs = eigvecs[:, ::-1]
    
    max_eig = eigvals[0]
    tau = max(tau_ratio * max_eig, min_tau)
    new_eigvals = np.maximum(eigvals, tau)
    F_reg = eigvecs @ np.diag(new_eigvals) @ eigvecs.T
    F_reg += ridge * np.eye(F.shape[0])
    return F_reg, eigvals, new_eigvals

# Пример использования на случайной матрице (не выполняем)

## 6. Байесовский вывод (обоснование целевой функции)

Теперь вернёмся к вероятностной интерпретации. Апостериорная плотность всех неизвестных при наблюдениях $y_{1:N}$:

$$
p(\theta, x_{0:N} \mid y_{1:N}) \propto p(\theta) \prod_{k=0}^{N-1} p(x_{k+1} \mid x_k, \theta) \prod_{k=1}^{N} p(y_k \mid x_k, \theta).
$$

Логарифмируя и подставляя гауссовские предположения, получаем:

$$
-\log p = \frac12\|\theta - \bar\theta\|_{P_\theta^{-1}}^2 +
\frac12\sum_{k=0}^{N-1}\|x_{k+1} - f(x_k,\theta)\|_{Q^{-1}}^2 +
\frac12\sum_{k=1}^{N}\|y_k - h(x_k,\theta)\|_{R^{-1}}^2 + \text{const}.
$$

Минимизация этого выражения даёт **максимальную апостериорную оценку (MAP)**. Именно её мы и решаем в MHE, но в скользящем окне, где априорные члены для состояния и параметров передаются из предыдущих окон.

## 7. Преимущества MHE с предложенной схемой

1. **Учёт ограничений** – в отличие от фильтра Калмана, можно явно задать границы состояний и параметров.
2. **Гибкое использование априорной информации** – arrival cost элегантно суммирует прошлые данные.
3. **Адаптивность** – forgetting factor позволяет отслеживать дрейф параметров.
4. **Оценка неопределённости** – накопленная FIM даёт ковариацию оценок, что важно для диагностики и планирования.

## 8. Заключение
Мы построили стройную теорию MHE от оптимизационной постановки до байесовского обоснования, с практическими деталями (регуляризация, выбор FIM, обновление точности). Представленный в репозитории код реализует все эти элементы и готов к экспериментам.

## Литература
1. Rao, Rawlings, Mayne (2003) – Constrained state estimation for nonlinear discrete-time systems.
2. Rawlings, Mayne (2009) – Model Predictive Control: Theory and Design.
3. Haseltine, Rawlings (2005) – Critical evaluation of EKF and MHE.
4. Kay (1993) – Fundamentals of Statistical Signal Processing: Estimation Theory.